# Model Deployment Preparation

## Objective

The objective of this notebook is to prepare the trained machine learning model
for deployment by loading all saved artifacts, verifying their integrity, and
creating reusable prediction functions.

This notebook does not perform model training. Instead, it focuses on making
the trained Gradient Boosting Regressor ready for integration into a production
environment such as a FastAPI backend.

## Workflow

- Load the trained model
- Load preprocessing artifacts
- Verify deployment files
- Build reusable prediction functions
- Perform single-house prediction
- Perform batch prediction
- Validate deployment readiness
- Summarize deployment status

## Expected Outcome

At the end of this notebook, all deployment artifacts will be verified and the
trained model will be fully prepared for backend integration and real-world
house price prediction.

# Configure Project Root

This notebook uses modules and model artifacts stored in different project
directories. To ensure consistent imports and file access, the project root
directory is automatically detected and added to Python's search path.

This configuration allows the notebook to:

- Import project modules.
- Access trained models.
- Load preprocessing artifacts.
- Maintain portability across different machines.

In [10]:
# ============================================================
# Configure Project Root
# ============================================================

import sys
from pathlib import Path

# Current notebook directory
NOTEBOOK_DIR = Path.cwd()

# Project root directory
PROJECT_ROOT = NOTEBOOK_DIR.parent

# Add project root to Python path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("=" * 70)
print("Project Root Configured Successfully")
print("=" * 70)

print(f"Notebook Directory : {NOTEBOOK_DIR}")
print(f"Project Root       : {PROJECT_ROOT}")


Project Root Configured Successfully
Notebook Directory : c:\Users\HP\AI Projects\Smart-House-Price-Prediction\notebooks
Project Root       : c:\Users\HP\AI Projects\Smart-House-Price-Prediction


# Import Required Libraries

This section imports all the libraries required for deployment preparation.
Unlike the model training notebook, only deployment-related libraries are
imported to keep the notebook lightweight and production-ready.

In [11]:
# ============================================================
# Import Required Libraries
# ============================================================

import warnings

import joblib
import numpy as np
import pandas as pd

from IPython.display import Markdown, display

from backend.utils.preprocessing_pipeline import (
    handle_missing_values,
    apply_ordinal_encoding,
    apply_nominal_encoding,
    preprocess_for_inference,
)

warnings.filterwarnings("ignore")

print("=" * 70)
print("Libraries Imported Successfully")
print("=" * 70)


Libraries Imported Successfully


# Load Trained Model

This section loads the trained Gradient Boosting Regressor that was saved
during the model training phase. Before loading, the notebook verifies that
the model file exists to prevent runtime errors during deployment.

In [12]:
# ============================================================
# Load Trained Model
# ============================================================

model_path = PROJECT_ROOT / "models" / "gradient_boosting_tuned.pkl"

if not model_path.exists():
    raise FileNotFoundError(
        f"Model not found:\n{model_path}"
    )

best_model = joblib.load(model_path)

display(Markdown("## 🤖 Trained Model"))

model_info = pd.DataFrame({
    "Property": [
        "Model Name",
        "Model Type",
        "Status",
        "Location"
    ],
    "Value": [
        model_path.stem,
        type(best_model).__name__,
        "Loaded Successfully ✅",
        str(model_path)
    ]
})

display(model_info.style.hide(axis="index"))

print("\n✅ Model loaded successfully.")


## 🤖 Trained Model

Property,Value
Model Name,gradient_boosting_tuned
Model Type,GradientBoostingRegressor
Status,Loaded Successfully ✅
Location,c:\Users\HP\AI Projects\Smart-House-Price-Prediction\models\gradient_boosting_tuned.pkl



✅ Model loaded successfully.


# Load Preprocessing Artifacts

This section loads all preprocessing artifacts that were saved during the data
preprocessing phase. These artifacts ensure that new input data undergoes the
same transformations as the training data, maintaining consistency between
training and deployment.

The loaded artifacts include:

- Preprocessing pipeline
- Standard scaler
- Feature column names
- Scaled columns
- Excluded columns
- Preprocessing metadata

In [13]:
# ============================================================
# Load Deployment Artifacts
# ============================================================

artifact_paths = {
    "Standard Scaler": PROJECT_ROOT / "models" / "standard_scaler.pkl",
    "Feature Columns": PROJECT_ROOT / "models" / "feature_columns.pkl",
    "Scaled Columns": PROJECT_ROOT / "models" / "scaled_columns.pkl",
    "Excluded Columns": PROJECT_ROOT / "models" / "excluded_columns.pkl",
    "Preprocessing Metadata": PROJECT_ROOT / "models" / "preprocessing_metadata.pkl",
}

loaded_artifacts = {}

for artifact_name, artifact_path in artifact_paths.items():

    if not artifact_path.exists():
        raise FileNotFoundError(
            f"{artifact_name} not found:\n{artifact_path}"
        )

    loaded_artifacts[artifact_name] = joblib.load(artifact_path)

display(Markdown("## 📦 Deployment Artifacts"))

artifact_summary = pd.DataFrame({
    "Artifact": artifact_paths.keys(),
    "Status": ["Loaded Successfully ✅"] * len(artifact_paths)
})

display(artifact_summary.style.hide(axis="index"))

print("\n✅ All deployment artifacts loaded successfully.")


## 📦 Deployment Artifacts

Artifact,Status
Standard Scaler,Loaded Successfully ✅
Feature Columns,Loaded Successfully ✅
Scaled Columns,Loaded Successfully ✅
Excluded Columns,Loaded Successfully ✅
Preprocessing Metadata,Loaded Successfully ✅



✅ All deployment artifacts loaded successfully.


# Verify Deployment Assets

Before deployment, all required assets should be verified to ensure they are
available and can be accessed successfully.

The verification process checks:

- Trained model
- Standard Scaler
- Feature Columns
- Scaled Columns
- Excluded Columns
- Preprocessing Metadata
- Preprocessing Pipeline Module

Successful verification indicates that the deployment environment is correctly
configured and ready for inference.

In [14]:
# ============================================================
# Verify Deployment Assets
# ============================================================

deployment_assets = {
    "Trained Model": PROJECT_ROOT / "models" / "gradient_boosting_tuned.pkl",
    "Standard Scaler": PROJECT_ROOT / "models" / "standard_scaler.pkl",
    "Feature Columns": PROJECT_ROOT / "models" / "feature_columns.pkl",
    "Scaled Columns": PROJECT_ROOT / "models" / "scaled_columns.pkl",
    "Excluded Columns": PROJECT_ROOT / "models" / "excluded_columns.pkl",
    "Preprocessing Metadata": PROJECT_ROOT / "models" / "preprocessing_metadata.pkl",
    "Preprocessing Pipeline": PROJECT_ROOT / "backend" / "utils" / "preprocessing_pipeline.py"
}

verification_results = []

print("=" * 70)
print("VERIFYING DEPLOYMENT ASSETS")
print("=" * 70)

for asset_name, asset_path in deployment_assets.items():

    exists = asset_path.exists()

    verification_results.append({
        "Asset": asset_name,
        "Status": "Available ✅" if exists else "Missing ❌",
        "Location": str(asset_path)
    })

verification_df = pd.DataFrame(verification_results)

display(Markdown("## 📦 Deployment Asset Verification"))

display(verification_df.style.hide(axis="index"))

available_assets = verification_df["Status"].str.contains("Available").sum()
total_assets = len(verification_df)

print("\n" + "=" * 70)
print(f"Deployment Assets Verified : {available_assets}/{total_assets}")

if available_assets == total_assets:
    print("Project is READY for deployment. ✅")
else:
    print("Some deployment assets are missing. ❌")

print("=" * 70)


VERIFYING DEPLOYMENT ASSETS


## 📦 Deployment Asset Verification

Asset,Status,Location
Trained Model,Available ✅,c:\Users\HP\AI Projects\Smart-House-Price-Prediction\models\gradient_boosting_tuned.pkl
Standard Scaler,Available ✅,c:\Users\HP\AI Projects\Smart-House-Price-Prediction\models\standard_scaler.pkl
Feature Columns,Available ✅,c:\Users\HP\AI Projects\Smart-House-Price-Prediction\models\feature_columns.pkl
Scaled Columns,Available ✅,c:\Users\HP\AI Projects\Smart-House-Price-Prediction\models\scaled_columns.pkl
Excluded Columns,Available ✅,c:\Users\HP\AI Projects\Smart-House-Price-Prediction\models\excluded_columns.pkl
Preprocessing Metadata,Available ✅,c:\Users\HP\AI Projects\Smart-House-Price-Prediction\models\preprocessing_metadata.pkl
Preprocessing Pipeline,Available ✅,c:\Users\HP\AI Projects\Smart-House-Price-Prediction\backend\utils\preprocessing_pipeline.py



Deployment Assets Verified : 7/7
Project is READY for deployment. ✅


# Create Prediction Function

This section defines a reusable prediction function that performs the complete
inference workflow.

The function:

- Validates the input data.
- Applies the preprocessing pipeline used during training.
- Aligns features with the trained model.
- Uses the saved StandardScaler for feature scaling.
- Predicts the house price using the trained Gradient Boosting model.
- Returns the prediction in a structured format.

This function can later be reused directly in the FastAPI backend without any
modification.

In [15]:
# ============================================================
# Create Prediction Function
# ============================================================

def predict_house_price(input_data, verbose=True):
    """
    Predict house price using the trained model.

    Parameters
    ----------
    input_data : dict or pandas.DataFrame
        Raw input features.

    verbose : bool, default=True
        Display prediction results.

    Returns
    -------
    dict
        Dictionary containing the predicted house price.
    """

    # --------------------------------------------------
    # Convert input to DataFrame
    # --------------------------------------------------

    if isinstance(input_data, dict):
        input_df = pd.DataFrame([input_data])

    elif isinstance(input_data, pd.DataFrame):
        input_df = input_data.copy()

    else:
        raise TypeError(
            "Input must be either a dictionary or a pandas DataFrame."
        )

    # --------------------------------------------------
    # Apply preprocessing pipeline
    # --------------------------------------------------

    processed_df = preprocess_for_inference(
        input_df=input_df,
        feature_columns=loaded_artifacts["Feature Columns"],
        scaler=loaded_artifacts["Standard Scaler"],
        scaled_columns=loaded_artifacts["Scaled Columns"],
        verbose=False
    )

    # --------------------------------------------------
    # Generate prediction
    # --------------------------------------------------

    predicted_price = float(best_model.predict(processed_df)[0])

    result = {
        "Predicted House Price": round(predicted_price, 2)
    }

    # --------------------------------------------------
    # Display results
    # --------------------------------------------------

    if verbose:

        display(Markdown("## 🏡 Prediction Result"))

        result_df = pd.DataFrame({
            "Output": ["Predicted House Price"],
            "Value": [f"${predicted_price:,.2f}"]
        })

        display(result_df.style.hide(axis="index"))

        print("\nPrediction completed successfully. ✅")

    return result


print("=" * 70)
print("PREDICTION FUNCTION CREATED SUCCESSFULLY")
print("=" * 70)


PREDICTION FUNCTION CREATED SUCCESSFULLY


# Single House Prediction

This section evaluates the deployment pipeline using a real house from the
training dataset.

The workflow includes:

- Loading a sample house.
- Separating the target value.
- Running the deployment preprocessing pipeline.
- Predicting the house price.
- Comparing the prediction with the actual selling price.

This serves as an end-to-end validation of the deployment pipeline.

In [16]:
# ============================================================
# Single House Prediction
# ============================================================

# Load training dataset
train_path = PROJECT_ROOT / "data" / "raw" / "train.csv"

train_df = pd.read_csv(train_path)

print("=" * 70)
print("TRAINING DATA LOADED")
print("=" * 70)
print(f"Dataset Shape : {train_df.shape}")

# --------------------------------------------------
# Select one sample
# --------------------------------------------------

sample_index = 0

sample_house = train_df.iloc[[sample_index]].copy()

actual_price = float(sample_house["SalePrice"].iloc[0])

sample_house.drop(columns=["SalePrice"], inplace=True)

print(f"\nSelected Sample Index : {sample_index}")
print(f"Actual House Price    : ${actual_price:,.2f}")

# --------------------------------------------------
# Predict
# --------------------------------------------------

prediction = predict_house_price(sample_house, verbose=False)

predicted_price = prediction["Predicted House Price"]

# --------------------------------------------------
# Calculate Error
# --------------------------------------------------

absolute_error = abs(actual_price - predicted_price)

percentage_error = (absolute_error / actual_price) * 100

# --------------------------------------------------
# Display Results
# --------------------------------------------------

display(Markdown("## 🏡 Single House Prediction"))

result_df = pd.DataFrame({
    "Metric": [
        "Actual Price",
        "Predicted Price",
        "Absolute Error",
        "Percentage Error"
    ],
    "Value": [
        f"${actual_price:,.2f}",
        f"${predicted_price:,.2f}",
        f"${absolute_error:,.2f}",
        f"{percentage_error:.2f}%"
    ]
})

display(result_df.style.hide(axis="index"))

print("\nPrediction completed successfully. ✅")


TRAINING DATA LOADED
Dataset Shape : (1460, 81)

Selected Sample Index : 0
Actual House Price    : $208,500.00
INSIDE apply_ordinal_encoding()

Checking category mappings...

✓ All categories successfully validated.

✓ Ordinal encoding completed successfully.

Encoded Columns:
ExterQual            -> int64
ExterCond            -> int64
BsmtQual             -> int64
BsmtCond             -> int64
HeatingQC            -> int64
KitchenQual          -> int64
FireplaceQu          -> int64
GarageQual           -> int64
GarageCond           -> int64
PoolQC               -> int64
BsmtExposure         -> int64
BsmtFinType1         -> int64
BsmtFinType2         -> int64
GarageFinish         -> int64
Functional           -> int64
INSIDE apply_nominal_encoding()

Nominal Features Detected : 28
  • MSZoning
  • Street
  • Alley
  • LotShape
  • LandContour
  • Utilities
  • LotConfig
  • LandSlope
  • Neighborhood
  • Condition1
  • Condition2
  • BldgType
  • HouseStyle
  • RoofStyle
  • RoofMatl
 

## 🏡 Single House Prediction

Metric,Value
Actual Price,"$208,500.00"
Predicted Price,"$202,245.44"
Absolute Error,"$6,254.56"
Percentage Error,3.00%



Prediction completed successfully. ✅


# 9. Batch House Price Prediction

This section demonstrates how the trained Gradient Boosting model can predict house prices for multiple properties simultaneously.

Batch inference is useful in real-world applications such as:

- Real estate valuation systems
- Banking and mortgage applications
- Property recommendation platforms
- Enterprise analytics

The same preprocessing pipeline used during training is automatically applied before generating predictions.

In [17]:
# ============================================================
# Batch House Price Prediction
# ============================================================

print("=" * 70)
print("BATCH HOUSE PRICE PREDICTION")
print("=" * 70)

# --------------------------------------------------
# Load Training Dataset
# --------------------------------------------------

train_df = pd.read_csv(train_path)

# --------------------------------------------------
# Select Multiple Samples
# --------------------------------------------------

sample_indices = [0, 25, 100, 250, 500]

batch_samples = train_df.iloc[sample_indices].copy()

actual_prices = batch_samples["SalePrice"].values

batch_input = batch_samples.drop(columns=["SalePrice"])

# --------------------------------------------------
# Predict Each House
# --------------------------------------------------

predictions = []

for idx, row in batch_input.iterrows():

    print(f"Predicting sample index: {idx}")

    try:
        prediction = predict_house_price(
            row.to_dict(),
            verbose=False
        )

        predictions.append(prediction["Predicted House Price"])

    except Exception as e:

        print(f"\nError at sample index: {idx}")
        print(e)

        raise

# --------------------------------------------------
# Create Results Table
# --------------------------------------------------

results_df = pd.DataFrame({
    "Sample Index": sample_indices,
    "Actual Price ($)": actual_prices,
    "Predicted Price ($)": predictions
})

results_df["Absolute Error ($)"] = (
    results_df["Actual Price ($)"] -
    results_df["Predicted Price ($)"]
).abs()

results_df["Percentage Error (%)"] = (
    results_df["Absolute Error ($)"] /
    results_df["Actual Price ($)"] * 100
).round(2)

# --------------------------------------------------
# Display Results
# --------------------------------------------------

print("\nBatch Prediction Completed Successfully!\n")

display(
    results_df.style.format({
        "Actual Price ($)": "${:,.2f}",
        "Predicted Price ($)": "${:,.2f}",
        "Absolute Error ($)": "${:,.2f}",
        "Percentage Error (%)": "{:.2f}%"
    })
)


BATCH HOUSE PRICE PREDICTION
Predicting sample index: 0
INSIDE apply_ordinal_encoding()

Checking category mappings...

✓ All categories successfully validated.

✓ Ordinal encoding completed successfully.

Encoded Columns:
ExterQual            -> int64
ExterCond            -> int64
BsmtQual             -> int64
BsmtCond             -> int64
HeatingQC            -> int64
KitchenQual          -> int64
FireplaceQu          -> int64
GarageQual           -> int64
GarageCond           -> int64
PoolQC               -> int64
BsmtExposure         -> int64
BsmtFinType1         -> int64
BsmtFinType2         -> int64
GarageFinish         -> int64
Functional           -> int64
INSIDE apply_nominal_encoding()

Nominal Features Detected : 28
  • MSZoning
  • Street
  • Alley
  • LotShape
  • LandContour
  • Utilities
  • LotConfig
  • LandSlope
  • Neighborhood
  • Condition1
  • Condition2
  • BldgType
  • HouseStyle
  • RoofStyle
  • RoofMatl
  • Exterior1st
  • Exterior2nd
  • MasVnrType
  • Found

,Sample Index,Actual Price ($),Predicted Price ($),Absolute Error ($),Percentage Error (%)
0,0,"$208,500.00","$202,245.44","$6,254.56",3.00%
1,25,"$256,300.00","$248,004.84","$8,295.16",3.24%
2,100,"$205,000.00","$202,681.52","$2,318.48",1.13%
3,250,"$76,500.00","$75,390.84","$1,109.16",1.45%
4,500,"$113,000.00","$109,846.73","$3,153.27",2.79%
